<a href="https://colab.research.google.com/github/fahadabdullah-lab/smap-dispatch-gee-south-texas/blob/main/notebooks/01_setup_auth_aoi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Cell 1 — Install & Imports**

In [ ]:
# Install required libraries
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

### **Cell 2 — Authenticate & Initialize Earth Engine**

In [ ]:
# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-fafahadabdullah')

print("✅ Earth Engine initialized")

### **Cell 3 — Define a working test window (10-day composite)**

In [ ]:
# ==================================================
# Processing configuration (10-day composite window)
# ==================================================

TEST_DATE = "2021-08-15"          # center date (YYYY-MM-DD)
COMPOSITE_DAYS = 10              # total window length
HALF_WINDOW = COMPOSITE_DAYS // 2  # +/- around TEST_DATE

center = ee.Date(TEST_DATE)
start  = center.advance(-HALF_WINDOW, "day")
end    = center.advance(HALF_WINDOW + 1, "day")   # +1 makes end exclusive, safer

print("📅 Center date:", TEST_DATE)
print("🪟 Composite window days:", COMPOSITE_DAYS)
print("➡️ Start:", start.format("YYYY-MM-dd").getInfo())
print("➡️ End  :", end.format("YYYY-MM-dd").getInfo())

# MODIS mode (we'll keep this switch for later)
USE_TERRA_ONLY = True
print("🌍 Terra-only mode:", USE_TERRA_ONLY)

### **Cell 4 — AOI (Draw polygon)**

In [ ]:
# ==================================================
# AOI — Draw polygon interactively
# ==================================================

import geemap

Map = geemap.Map(center=[26.1, -97.3], zoom=9)

# add satellite basemap
Map.add_basemap("HYBRID")

# enable drawing tool
Map.add_draw_control()

print("✏️ Use the drawing tool on the left panel to draw your AOI polygon.")
print("After drawing, run the next cell to convert it to an Earth Engine geometry.")

Map

### **Cell 5 — Convert drawn polygon**

In [ ]:
# ==================================================
# Cell 5 — Convert drawn geometry to AOI (list-safe)
# ==================================================
import ee

drawn_obj = None

# Try common places where geemap stores drawings
if hasattr(Map, "draw_features") and Map.draw_features is not None:
    drawn_obj = Map.draw_features
elif hasattr(Map, "drawn_features") and Map.drawn_features is not None:
    drawn_obj = Map.drawn_features
elif hasattr(Map, "draw_control") and Map.draw_control is not None:
    try:
        drawn_obj = Map.draw_control.data  # often a Python list of GeoJSON features
    except Exception:
        drawn_obj = None

if drawn_obj is None:
    raise ValueError("❌ Could not find drawn geometry. Re-run the draw cell, draw polygon, then run again.")

# --- Normalize to an EE FeatureCollection ---
if isinstance(drawn_obj, list):
    # GeoJSON features list -> EE FeatureCollection
    if len(drawn_obj) == 0:
        raise ValueError("❌ No polygon found. Please draw a polygon first.")
    drawn_fc = ee.FeatureCollection(drawn_obj)

else:
    # Already an EE FeatureCollection (or something EE-like)
    drawn_fc = drawn_obj

# --- Validate and extract AOI ---
n = drawn_fc.size().getInfo()
print("✅ Drawn feature count:", n)

if n == 0:
    raise ValueError("❌ No polygon found. Please draw a polygon first (use the polygon tool).")

aoi = ee.Feature(drawn_fc.first()).geometry()
print("✅ AOI loaded from drawn polygon")

### **CELL 6 — Visualize AOI**

In [ ]:
Map2 = geemap.Map()
Map2.add_basemap("HYBRID")
Map2.addLayer(aoi, {"color": "yellow"}, "AOI (drawn)")
Map2.centerObject(aoi, 10)
Map2

### **CELL 7 — Quick dataset sanity check**

In [ ]:
# Quick dataset sanity check (MODIS LST, using composite window)

modis_lst = (
    ee.ImageCollection("MODIS/061/MOD11A1")
    .filterDate(start, end)
    .filterBounds(aoi)
)

print("MOD11A1 images found (window):", modis_lst.size().getInfo())

In [ ]:
# SMAP availability check (window)
smap_ic = (
    ee.ImageCollection("NASA_USDA/HSL/SMAP10KM_soil_moisture")
    .filterDate(start, end)
    .filterBounds(aoi)
)

print("SMAP10KM images found (window):", smap_ic.size().getInfo())